In [29]:
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm

In [30]:
np.random.seed(0)

In [31]:
def bayesian_update(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [ ]:
def best_response(x, thresholds, priors, c):
    posteriors = bayesian_update(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for x_p in search_space:
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost)
    return search_space[np.argmax(utilities)]

In [33]:
def balance_priors(priors, random=True):
    total = np.sum(priors)
    if total == 1:
        return priors
    indices = priors == 0
    remainder = 1 - total
    if random:
        p = np.random.rand(indices.sum())
        p = (p / p.sum()) * remainder
    else:
        p = remainder / indices.sum()
    priors[indices] = p
    return priors

In [ ]:
def accuracy_loss(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    losses = []
    for threshold in thresholds:
        Y_p = (X_p >= threshold).astype(float)
        acc_loss = np.abs(Y_true - Y_p).mean()
        losses.append(acc_loss)
    losses = np.array(losses)
    posteriors = bayesian_update(priors)
    return np.dot(losses, posteriors)

In [ ]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    threshold_p = thresholds[partition]
    priors_p = priors[partition]
    X_p = []
    for x in X:
        x_p = best_response(x, threshold_p, priors_p, c)
        X_p.append(x_p)
    X_p = np.array(X_p)
    acc_loss = accuracy_loss(X, X_p, threshold_p, threshold_true)
    return priors_p, acc_loss

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.
    for partition in partitions:
        priors_p, acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += np.dot(priors_p, acc_loss_p)
    return acc_loss

In [36]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [37]:
def find_partitions_greedy(X, thresholds, priors, threshold_true, c):
    partitions = [[i] for i in range(len(priors))]
    P = {}
    next_id = 0

    P_init = [list(block) for block in partitions]
    for block in P_init:
        P[next_id] = list(block)
        next_id += 1

    active_ids = set(P.keys())

    Q = collections.deque(itertools.combinations(active_ids, 2))

    while Q:
        a_id, b_id = Q.popleft()

        # Skip if one was merged already
        if a_id not in active_ids or b_id not in active_ids:
            continue

        a = P[a_id]
        b = P[b_id]

        priors_a, acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        priors_b, acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

        lhs = np.dot(priors_a, acc_loss_a) + np.dot(priors_b, acc_loss_b)
        ab = sorted(a + b)
        priors_ab, acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        rhs = np.dot(priors_ab, acc_loss_ab)

        if lhs > rhs:
            # -------- MERGE --------
            # Remove a and b from partition
            active_ids.remove(a_id)
            active_ids.remove(b_id)
            del P[a_id]
            del P[b_id]

            # Remove all (a, c) and (b, d) from Q
            Q = collections.deque(
                (x, y)
                for (x, y) in Q
                if x not in {a_id, b_id} and y not in {a_id, b_id}
            )

            # Add merged block
            new_id = next_id
            next_id += 1
            P[new_id] = ab
            active_ids.add(new_id)

            # Insert (ab, c) for all remaining c in P \ {a,b}
            for c_id in active_ids:
                if c_id != new_id:
                    Q.append((new_id, c_id))
    return list(P.values())

In [38]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = [i for i in range(len(thresholds))]

    parts = set_partitions(indices)
    partitions_set = []
    for part in parts:
        partitions_set.append(part)


    best_partition = None
    best_loss = np.inf

    for partitions in partitions_set:
        acc_loss = 0.
        for partition in partitions:
            priors_p, acc_loss_p = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
            acc_loss += np.dot(priors_p, acc_loss_p)

        if acc_loss < best_loss:
            best_loss = acc_loss
            best_partition = partitions

    return best_partition

In [39]:
c = 5.0
threshold_true = 0.5

threshold_min, threshold_max, threshold_delta = 0., 1., 0.1
thresholds = np.arange(threshold_min+threshold_delta, threshold_max, threshold_delta).round(4)
X = np.arange(threshold_min, threshold_max, 0.001).round(4)
print(len(X))

# thresholds = np.array([0.2, 0.25, 0.8])

# priors = np.zeros_like(thresholds)
# priors[0] = 0.3
# priors[1] = 0.3
# priors[2] = 0.4

priors = np.array([0.11393634, 0.11459784, 0.03594993, 0.25, 0.02203078, 0.05390004, 0.06215049, 0.09743458, 0.25])
# balance_priors(priors, random=True)

print(np.sum(priors))
pd.DataFrame({"threshold": thresholds, "priors": priors}).round(3).T

1000
1.0


,0,1,2,3,4,5,6,7,8
threshold,0.100,0.200,0.300,0.40,0.500,0.600,0.700,0.800,0.90
priors,0.114,0.115,0.036,0.25,0.022,0.054,0.062,0.097,0.25


In [40]:
partition_greedy = find_partitions_greedy(X, thresholds, priors, threshold_true, c)
partition_optimal = find_partitions_optimal(X, thresholds, priors, threshold_true, c)

acc_loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
acc_loss_optimal = evaluate_system(X, partition_optimal, thresholds, priors, threshold_true, c)

In [41]:
print("Greedy")
print("------")
print(f"Partition: {sorted(partition_greedy)}")
print(f"Acc Loss : {acc_loss_greedy:.4f}")
print()
print("Optimal")
print("-------")
print(f"Partition: {sorted(partition_optimal)}")
print(f"Acc Loss : {acc_loss_optimal:.4f}")

Greedy
------
Partition: [[0, 1, 2, 3, 4, 5, 6, 8], [7]]
Acc Loss : 0.2445

Optimal
-------
Partition: [[0, 1, 2, 3, 4], [5, 6], [7], [8]]
Acc Loss : 0.2057
